<a href="https://colab.research.google.com/github/georgmandryka-cyber/kundennachrichten-sortieren/blob/main/nachrichten_sortieren.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install anthropic -q

from google.colab import userdata
import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def frage_claude(frage):
    antwort = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=2000,
        system="Antworte auf Deutsch, kurz und praktisch, maximal 5 Sätze, ohne Tabellen.",
        messages=[{"role": "user", "content": frage}]
    )
    teile = [block.text for block in antwort.content if block.type == "text"]
    return "".join(teile)

# 1. Beispieldatei erstellen (einmalig)
nachrichten = [
    "Das Produkt ist super, danke!",
    "Meine Bestellung ist nach zwei Wochen noch nicht da.",
    "Wie kann ich meine Adresse ändern?",
    "Na toll, schon wieder kaputt.",
    "Könnt ihr mir bitte sagen, wann ihr öffnet? Euer Laden ist sonst top!",
    "Ich wollte nur sagen, dass ich umgezogen bin."
]
pd.DataFrame({"nachricht": nachrichten}).to_csv("eingang.csv", index=False)

# 2. Datei laden und einordnen
df = pd.read_csv("eingang.csv")
df["kategorie"] = df["nachricht"].apply(
    lambda text: frage_claude(
        f"Ordne diese Nachricht genau einer Kategorie zu: Lob, Beschwerde, Frage oder Sonstiges. "
        f"Bei gemischten Nachrichten zählt das Hauptanliegen. "
        f"Antworte nur mit dem Wort. Nachricht: {text}"
    )
)

# 3. Ergebnis speichern
df.to_csv("ausgang.csv", index=False)
print(df)

                                           nachricht   kategorie
0                      Das Produkt ist super, danke!         Lob
1  Meine Bestellung ist nach zwei Wochen noch nic...  Beschwerde
2                 Wie kann ich meine Adresse ändern?       Frage
3                      Na toll, schon wieder kaputt.  Beschwerde
4  Könnt ihr mir bitte sagen, wann ihr öffnet? Eu...       Frage
5      Ich wollte nur sagen, dass ich umgezogen bin.   Sonstiges
